# Google Play Weak Sentiment Error Analysis and Cross-App Transfer v1

## Objective

This phase follows the rating-derived weak-sentiment baseline and answers three
focused questions from John:

1. What kinds of reviews are producing the baseline errors, especially for
   three-star reviews?
2. Do the engineered review features add useful signal beyond TF-IDF alone?
3. Does the current model transfer to apps that were completely absent from
   training?

The goal is diagnosis rather than a higher headline accuracy. The notebook keeps
the prior label mapping, text redaction, grouped split, model family, and fixed
hyperparameters so that the comparisons remain controlled.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
LABEL_ORDER = ["negative", "neutral", "positive"]
JOHN_CATEGORY_ORDER = [
    "neutral",
    "mixed",
    "positive",
    "negative",
    "unclear",
    "inconsistent_with_rating",
]
HELD_OUT_APPS = ["YouTube", "DoorDash"]

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
INPUT_DIR = REPO_ROOT / "inputs"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORT_DIR = REPO_ROOT / "reports"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MODELING_PATH = OUTPUT_DIR / "modeling_ready_weak_sentiment_v0.csv"
FEATURE_PATH = OUTPUT_DIR / "review_features_v0.csv"
ANNOTATION_PATH = INPUT_DIR / "manual_error_annotations_v1.csv"

EXPECTED_MODELING_SHA256 = (
    "53fc5b63a797ba22a063ca6c67a5062d8f3d84441112ee583dd0d997ca1405ee"
)
EXPECTED_FEATURE_SHA256 = (
    "7e65a3d282b7121012a940c931bed0a10f32ebf83d8744d1021545dbb87fe2fc"
)

TEXT_COLUMN = "model_text"
NUMERIC_COLUMNS = [
    "review_char_count",
    "review_word_count",
    "alphanumeric_char_count",
    "same_app_text_frequency",
    "issue_indicator_count",
]
BINARY_COLUMNS = [
    "rating_mention_redacted_flag",
    "short_review_flag",
    "low_signal_flag",
    "repeated_text_flag",
    "issue_crash_bug_flag",
    "issue_performance_loading_flag",
    "issue_login_account_flag",
    "issue_payment_billing_flag",
    "issue_ads_flag",
    "issue_update_version_flag",
    "issue_support_service_flag",
    "any_issue_indicator_flag",
]
CATEGORICAL_COLUMNS = ["language_group"]


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


print("Repository root:", REPO_ROOT)
print("Random state:", RANDOM_STATE)
print("Held-out apps:", HELD_OUT_APPS)


Repository root: /workspace/scratch/b0526b08776c/phase3_work/build/google_play_weak_sentiment_error_analysis_v1_complete_github_package
Random state: 42
Held-out apps: ['YouTube', 'DoorDash']


## 1. Source validation and continuity checks

The analysis uses the exact 34,601-row modeling-ready table from baseline v0,
the exact validated feature table used to create it, and a reviewer-authored
annotation file. Original cleaned text and source scores are used only for the
manual review. Models continue to use `model_text`, where direct written star
expressions were redacted.


In [2]:
for required_path in [MODELING_PATH, FEATURE_PATH, ANNOTATION_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required input not found: {required_path}")

modeling = pd.read_csv(MODELING_PATH)
source = pd.read_csv(
    FEATURE_PATH,
    usecols=[
        "review_key",
        "review_id",
        "content_cleaned",
        "score",
        "rating_group",
    ],
)
manual_annotations = pd.read_csv(ANNOTATION_PATH)

modeling["model_text"] = modeling["model_text"].fillna("").astype(str)
source["content_cleaned"] = source["content_cleaned"].fillna("").astype(str)

source_checks = [
    {
        "check": "modeling_file_sha256_matches_baseline_v0",
        "passed": sha256_file(MODELING_PATH) == EXPECTED_MODELING_SHA256,
        "observed": sha256_file(MODELING_PATH),
    },
    {
        "check": "feature_file_sha256_matches_feature_engineering_v0",
        "passed": sha256_file(FEATURE_PATH) == EXPECTED_FEATURE_SHA256,
        "observed": sha256_file(FEATURE_PATH),
    },
    {
        "check": "modeling_rows_equal_34601",
        "passed": len(modeling) == 34601,
        "observed": len(modeling),
    },
    {
        "check": "feature_rows_equal_34601",
        "passed": len(source) == 34601,
        "observed": len(source),
    },
    {
        "check": "modeling_review_keys_unique",
        "passed": modeling["review_key"].is_unique,
        "observed": int(modeling["review_key"].duplicated().sum()),
    },
    {
        "check": "feature_review_keys_unique",
        "passed": source["review_key"].is_unique,
        "observed": int(source["review_key"].duplicated().sum()),
    },
]
source_validation = pd.DataFrame(source_checks)
source_validation.to_csv(
    OUTPUT_DIR / "weak_sentiment_error_analysis_source_validation_v1.csv",
    index=False,
)

if not source_validation["passed"].all():
    failed = source_validation.loc[~source_validation["passed"]]
    raise ValueError("Source validation failed:\n" + failed.to_string(index=False))

data = modeling.merge(
    source,
    on="review_key",
    how="left",
    validate="one_to_one",
)
if len(data) != 34601 or data["score"].isna().any():
    raise ValueError("The modeling and source tables did not merge one-to-one.")

expected_labels = data["score"].map(
    {
        1: "negative",
        2: "negative",
        3: "neutral",
        4: "positive",
        5: "positive",
    }
)
if not expected_labels.equals(data["weak_sentiment_label"]):
    raise ValueError("Weak labels do not match the documented score mapping.")

train_df = data.loc[data["dataset_split"].eq("train")].copy()
test_df = data.loc[data["dataset_split"].eq("test")].copy()
baseline_text_overlap = len(
    set(train_df["text_group_id"]).intersection(test_df["text_group_id"])
)
if baseline_text_overlap != 0:
    raise ValueError("Baseline train/test text-group overlap is not zero.")

print(source_validation.to_string(index=False))
print()
print(f"Merged review rows: {len(data):,}")
print(f"Baseline train/test rows: {len(train_df):,} / {len(test_df):,}")
print("Baseline train/test text-group overlap:", baseline_text_overlap)


                                             check  passed                                                         observed
          modeling_file_sha256_matches_baseline_v0    True 53fc5b63a797ba22a063ca6c67a5062d8f3d84441112ee583dd0d997ca1405ee
feature_file_sha256_matches_feature_engineering_v0    True 7e65a3d282b7121012a940c931bed0a10f32ebf83d8744d1021545dbb87fe2fc
                         modeling_rows_equal_34601    True                                                            34601
                          feature_rows_equal_34601    True                                                            34601
                       modeling_review_keys_unique    True                                                                0
                        feature_review_keys_unique    True                                                                0

Merged review rows: 34,601
Baseline train/test rows: 27,681 / 6,920
Baseline train/test text-group overlap: 0


## 2. Controlled feature-set comparison

Both models use the same training rows, test rows, redacted text, TF-IDF
configuration, class weighting, `LinearSVC(C=1.0)`, and random state.

- `tfidf_only`: word unigrams and bigrams only
- `tfidf_plus_engineered_features`: the current baseline feature set

No hyperparameter search is performed. This isolates whether the engineered
features contribute under the existing evaluation setup.


In [3]:
def make_tfidf():
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.98,
        max_features=6000,
        sublinear_tf=True,
    )


def make_full_model():
    preprocessor = ColumnTransformer(
        transformers=[
            ("text", make_tfidf(), TEXT_COLUMN),
            (
                "numeric",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                NUMERIC_COLUMNS,
            ),
            (
                "binary",
                SimpleImputer(strategy="most_frequent"),
                BINARY_COLUMNS,
            ),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_COLUMNS,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            (
                "model",
                LinearSVC(
                    C=1.0,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def make_text_only_model():
    return Pipeline(
        steps=[
            ("tfidf", make_tfidf()),
            (
                "model",
                LinearSVC(
                    C=1.0,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def metric_row(experiment, model_name, y_true, y_pred):
    return {
        "experiment": experiment,
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "test_rows": len(y_true),
    }


def class_metric_rows(experiment, model_name, y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        output_dict=True,
        zero_division=0,
    )
    rows = []
    for label in LABEL_ORDER:
        rows.append(
            {
                "experiment": experiment,
                "model": model_name,
                "class": label,
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1_score": report[label]["f1-score"],
                "support": int(report[label]["support"]),
            }
        )
    return rows


full_model = make_full_model()
full_model.fit(train_df, train_df["weak_sentiment_label"])
full_predictions = full_model.predict(test_df)

text_only_model = make_text_only_model()
text_only_model.fit(train_df[TEXT_COLUMN], train_df["weak_sentiment_label"])
text_only_predictions = text_only_model.predict(test_df[TEXT_COLUMN])

feature_comparison_metrics = pd.DataFrame(
    [
        metric_row(
            "same_grouped_split",
            "tfidf_only",
            test_df["weak_sentiment_label"],
            text_only_predictions,
        ),
        metric_row(
            "same_grouped_split",
            "tfidf_plus_engineered_features",
            test_df["weak_sentiment_label"],
            full_predictions,
        ),
    ]
)
feature_comparison_class_metrics = pd.DataFrame(
    class_metric_rows(
        "same_grouped_split",
        "tfidf_only",
        test_df["weak_sentiment_label"],
        text_only_predictions,
    )
    + class_metric_rows(
        "same_grouped_split",
        "tfidf_plus_engineered_features",
        test_df["weak_sentiment_label"],
        full_predictions,
    )
)

comparison_predictions = test_df[
    [
        "review_key",
        "app_name",
        "score",
        "weak_sentiment_label",
        "model_text",
    ]
].copy()
comparison_predictions["tfidf_only_prediction"] = text_only_predictions
comparison_predictions["full_feature_prediction"] = full_predictions
comparison_predictions["tfidf_only_correct"] = (
    comparison_predictions["tfidf_only_prediction"]
    == comparison_predictions["weak_sentiment_label"]
)
comparison_predictions["full_feature_correct"] = (
    comparison_predictions["full_feature_prediction"]
    == comparison_predictions["weak_sentiment_label"]
)

agreement_count = int(
    (
        comparison_predictions["tfidf_only_prediction"]
        == comparison_predictions["full_feature_prediction"]
    ).sum()
)
comparison_change_summary = pd.DataFrame(
    [
        {
            "measure": "prediction_agreement_count",
            "value": agreement_count,
        },
        {
            "measure": "prediction_agreement_rate",
            "value": agreement_count / len(comparison_predictions),
        },
        {
            "measure": "predictions_changed",
            "value": len(comparison_predictions) - agreement_count,
        },
        {
            "measure": "both_models_correct",
            "value": int(
                (
                    comparison_predictions["tfidf_only_correct"]
                    & comparison_predictions["full_feature_correct"]
                ).sum()
            ),
        },
        {
            "measure": "full_features_only_correct",
            "value": int(
                (
                    ~comparison_predictions["tfidf_only_correct"]
                    & comparison_predictions["full_feature_correct"]
                ).sum()
            ),
        },
        {
            "measure": "tfidf_only_correct",
            "value": int(
                (
                    comparison_predictions["tfidf_only_correct"]
                    & ~comparison_predictions["full_feature_correct"]
                ).sum()
            ),
        },
        {
            "measure": "both_models_wrong",
            "value": int(
                (
                    ~comparison_predictions["tfidf_only_correct"]
                    & ~comparison_predictions["full_feature_correct"]
                ).sum()
            ),
        },
    ]
)

feature_comparison_metrics.to_csv(
    OUTPUT_DIR / "weak_sentiment_feature_set_comparison_metrics_v1.csv",
    index=False,
)
feature_comparison_class_metrics.to_csv(
    OUTPUT_DIR / "weak_sentiment_feature_set_class_metrics_v1.csv",
    index=False,
)
comparison_change_summary.to_csv(
    OUTPUT_DIR / "weak_sentiment_feature_set_prediction_changes_v1.csv",
    index=False,
)

print(feature_comparison_metrics.round(6).to_string(index=False))
print()
print(feature_comparison_class_metrics.round(6).to_string(index=False))
print()
print(comparison_change_summary.to_string(index=False))


        experiment                          model  accuracy  balanced_accuracy  macro_f1  weighted_f1  test_rows
same_grouped_split                     tfidf_only  0.821098           0.585910  0.589228     0.820225       6920
same_grouped_split tfidf_plus_engineered_features  0.820809           0.586718  0.589886     0.819626       6920

        experiment                          model    class  precision   recall  f1_score  support
same_grouped_split                     tfidf_only negative   0.783741 0.741206  0.761880     1990
same_grouped_split                     tfidf_only  neutral   0.109756 0.110769  0.110260      325
same_grouped_split                     tfidf_only positive   0.885563 0.905755  0.895545     4605
same_grouped_split tfidf_plus_engineered_features negative   0.779598 0.741206  0.759918     1990
same_grouped_split tfidf_plus_engineered_features  neutral   0.115265 0.113846  0.114551      325
same_grouped_split tfidf_plus_engineered_features positive   0.885490 0.

## 3. Manual review of baseline errors

The baseline made 1,240 errors on the 6,920-row grouped test split.

The review design has two layers:

- **Balanced core:** 90 errors with 30 from each weak-label class. Within each
  class, 15 are selected from each wrong prediction direction.
- **Extra three-star review:** 30 additional neutral-label errors with 15 from
  each wrong prediction direction.

Within each error direction, rows are shuffled reproducibly and selected in an
app round-robin order. The final sample has 120 unique reviews: 30 negative,
60 neutral, and 30 positive weak labels.

One reviewer reads the original cleaned text and source rating and records:

- text sentiment: neutral, mixed, positive, negative, or unclear
- rating consistency: consistent, partially inconsistent, inconsistent, or unclear
- one primary category using John's requested six-category vocabulary

`inconsistent_with_rating` is used only when a clear text interpretation
conflicts with the rating-derived weak label. Counts describe this designed
error sample and are not population estimates for all reviews.


In [4]:
def round_robin_app_sample(frame, n, seed):
    if len(frame) < n:
        raise ValueError(f"Requested {n} rows from only {len(frame)} candidates.")
    shuffled = frame.sample(frac=1, random_state=seed).copy()
    shuffled["_within_app_order"] = shuffled.groupby("app_name").cumcount()
    return (
        shuffled.sort_values(
            ["_within_app_order", "app_name", "review_key"],
            kind="stable",
        )
        .head(n)
        .drop(columns="_within_app_order")
    )


error_pool = comparison_predictions.loc[
    ~comparison_predictions["full_feature_correct"]
].merge(
    source[["review_key", "review_id", "content_cleaned", "rating_group"]],
    on="review_key",
    how="left",
    validate="one_to_one",
)
error_pool["error_direction"] = (
    error_pool["weak_sentiment_label"]
    + "_to_"
    + error_pool["full_feature_prediction"]
)

sample_plan = [
    ("negative", "neutral", 15, "core_balanced", 101),
    ("negative", "positive", 15, "core_balanced", 102),
    ("neutral", "negative", 15, "core_balanced", 103),
    ("neutral", "positive", 15, "core_balanced", 104),
    ("positive", "negative", 15, "core_balanced", 105),
    ("positive", "neutral", 15, "core_balanced", 106),
    ("neutral", "negative", 15, "neutral_oversample", 203),
    ("neutral", "positive", 15, "neutral_oversample", 204),
]

sample_parts = []
used_keys = set()
for actual, predicted, count, layer, seed in sample_plan:
    candidates = error_pool.loc[
        error_pool["weak_sentiment_label"].eq(actual)
        & error_pool["full_feature_prediction"].eq(predicted)
        & ~error_pool["review_key"].isin(used_keys)
    ].copy()
    selected = round_robin_app_sample(candidates, count, seed)
    selected["sample_layer"] = layer
    sample_parts.append(selected)
    used_keys.update(selected["review_key"])

manual_sample = pd.concat(sample_parts, ignore_index=True)
manual_sample = manual_sample.sort_values(
    [
        "sample_layer",
        "weak_sentiment_label",
        "full_feature_prediction",
        "app_name",
        "review_key",
    ],
    kind="stable",
).reset_index(drop=True)
manual_sample.insert(
    0,
    "sample_id",
    [f"EA{i:03d}" for i in range(1, len(manual_sample) + 1)],
)

annotation_required_columns = {
    "sample_id",
    "review_key",
    "manual_text_category",
    "rating_consistency",
    "john_primary_category",
    "annotation_rationale",
}
if not annotation_required_columns.issubset(manual_annotations.columns):
    missing = sorted(annotation_required_columns - set(manual_annotations.columns))
    raise ValueError(f"Manual annotation columns missing: {missing}")

annotated_errors = manual_sample.merge(
    manual_annotations,
    on=["sample_id", "review_key"],
    how="left",
    validate="one_to_one",
)
if annotated_errors["john_primary_category"].isna().any():
    raise ValueError("At least one sampled error is missing a manual annotation.")

valid_text_categories = {"neutral", "mixed", "positive", "negative", "unclear"}
valid_consistency = {
    "consistent",
    "partially_inconsistent",
    "inconsistent",
    "unclear",
}
if not set(annotated_errors["manual_text_category"]).issubset(
    valid_text_categories
):
    raise ValueError("Unexpected manual text category.")
if not set(annotated_errors["rating_consistency"]).issubset(valid_consistency):
    raise ValueError("Unexpected rating-consistency category.")
if not set(annotated_errors["john_primary_category"]).issubset(
    set(JOHN_CATEGORY_ORDER)
):
    raise ValueError("Unexpected John primary category.")

manual_category_summary = (
    annotated_errors["john_primary_category"]
    .value_counts()
    .reindex(JOHN_CATEGORY_ORDER, fill_value=0)
    .rename_axis("john_primary_category")
    .reset_index(name="review_count")
)
manual_category_summary["review_share"] = (
    manual_category_summary["review_count"] / len(annotated_errors)
)

label_category_index = pd.MultiIndex.from_product(
    [LABEL_ORDER, JOHN_CATEGORY_ORDER],
    names=["weak_sentiment_label", "john_primary_category"],
)
manual_by_weak_label = (
    annotated_errors.groupby(
        ["weak_sentiment_label", "john_primary_category"]
    )
    .size()
    .reindex(label_category_index, fill_value=0)
    .rename("review_count")
    .reset_index()
)
manual_by_weak_label["within_weak_label_share"] = (
    manual_by_weak_label["review_count"]
    / manual_by_weak_label.groupby("weak_sentiment_label")[
        "review_count"
    ].transform("sum")
)

manual_text_by_weak_label = (
    annotated_errors.groupby(
        ["weak_sentiment_label", "manual_text_category"]
    )
    .size()
    .rename("review_count")
    .reset_index()
)
manual_consistency_summary = (
    annotated_errors.groupby(
        ["weak_sentiment_label", "rating_consistency"]
    )
    .size()
    .rename("review_count")
    .reset_index()
)
manual_direction_summary = (
    annotated_errors.groupby(
        [
            "sample_layer",
            "error_direction",
            "john_primary_category",
        ]
    )
    .size()
    .rename("review_count")
    .reset_index()
)

annotated_errors.to_csv(
    OUTPUT_DIR / "weak_sentiment_manual_error_review_v1.csv",
    index=False,
)
manual_category_summary.to_csv(
    OUTPUT_DIR / "weak_sentiment_manual_error_category_summary_v1.csv",
    index=False,
)
manual_by_weak_label.to_csv(
    OUTPUT_DIR / "weak_sentiment_manual_error_by_weak_label_v1.csv",
    index=False,
)
manual_text_by_weak_label.to_csv(
    OUTPUT_DIR / "weak_sentiment_manual_text_by_weak_label_v1.csv",
    index=False,
)
manual_consistency_summary.to_csv(
    OUTPUT_DIR / "weak_sentiment_manual_rating_consistency_v1.csv",
    index=False,
)
manual_direction_summary.to_csv(
    OUTPUT_DIR / "weak_sentiment_manual_error_by_direction_v1.csv",
    index=False,
)

print(f"Baseline test errors: {len(error_pool):,}")
print(f"Manually reviewed errors: {len(annotated_errors):,}")
print()
print(manual_category_summary.to_string(index=False))
print()
print(
    manual_by_weak_label.pivot(
        index="weak_sentiment_label",
        columns="john_primary_category",
        values="review_count",
    )
    .fillna(0)
    .astype(int)
    .to_string()
)


Baseline test errors: 1,240
Manually reviewed errors: 120

   john_primary_category  review_count  review_share
                 neutral             0      0.000000
                   mixed            23      0.191667
                positive            10      0.083333
                negative            17      0.141667
                 unclear            18      0.150000
inconsistent_with_rating            52      0.433333

john_primary_category  inconsistent_with_rating  mixed  negative  neutral  positive  unclear
weak_sentiment_label                                                                        
negative                                      2      5        17        0         0        6
neutral                                      40     13         0        0         0        7
positive                                     10      5         0        0        10        5


## 4. Two-app holdout transfer test

YouTube and DoorDash are held out together. They were chosen before fitting
because they provide a deliberate contrast in app domain, review volume, and
prior collection activity while both contain all three weak-label classes.

The primary transfer test is strict:

- neither app appears in training
- every exact normalized-text group appearing in either held-out app is also
  removed from training
- all held-out reviews remain in the test set

A disclosed app-only sensitivity run keeps all eight-app training rows even
when generic exact text also appears in a held-out app. The strict result is
the main result; the sensitivity run checks whether shared generic text changes
the conclusion.


In [5]:
holdout_test = data.loc[data["app_name"].isin(HELD_OUT_APPS)].copy()
app_only_train = data.loc[~data["app_name"].isin(HELD_OUT_APPS)].copy()

heldout_text_groups = set(holdout_test["text_group_id"])
strict_holdout_train = app_only_train.loc[
    ~app_only_train["text_group_id"].isin(heldout_text_groups)
].copy()

app_only_overlap_groups = len(
    set(app_only_train["text_group_id"]).intersection(
        holdout_test["text_group_id"]
    )
)
app_only_overlap_train_rows = int(
    app_only_train["text_group_id"].isin(heldout_text_groups).sum()
)
app_only_train_groups = set(app_only_train["text_group_id"])
app_only_overlap_test_rows = int(
    holdout_test["text_group_id"].isin(app_only_train_groups).sum()
)
strict_overlap_groups = len(
    set(strict_holdout_train["text_group_id"]).intersection(
        holdout_test["text_group_id"]
    )
)

strict_holdout_model = make_full_model()
strict_holdout_model.fit(
    strict_holdout_train,
    strict_holdout_train["weak_sentiment_label"],
)
strict_holdout_predictions = strict_holdout_model.predict(holdout_test)

app_only_model = make_full_model()
app_only_model.fit(app_only_train, app_only_train["weak_sentiment_label"])
app_only_predictions = app_only_model.predict(holdout_test)

holdout_metrics = pd.DataFrame(
    [
        metric_row(
            "two_app_holdout",
            "strict_app_and_text_group_holdout",
            holdout_test["weak_sentiment_label"],
            strict_holdout_predictions,
        ),
        metric_row(
            "two_app_holdout_sensitivity",
            "app_only_holdout_with_disclosed_text_overlap",
            holdout_test["weak_sentiment_label"],
            app_only_predictions,
        ),
    ]
)
holdout_class_metrics = pd.DataFrame(
    class_metric_rows(
        "two_app_holdout",
        "strict_app_and_text_group_holdout",
        holdout_test["weak_sentiment_label"],
        strict_holdout_predictions,
    )
    + class_metric_rows(
        "two_app_holdout_sensitivity",
        "app_only_holdout_with_disclosed_text_overlap",
        holdout_test["weak_sentiment_label"],
        app_only_predictions,
    )
)

holdout_predictions_frame = holdout_test[
    [
        "review_key",
        "app_name",
        "score",
        "weak_sentiment_label",
    ]
].copy()
holdout_predictions_frame["strict_holdout_prediction"] = (
    strict_holdout_predictions
)
holdout_predictions_frame["strict_holdout_correct"] = (
    holdout_predictions_frame["strict_holdout_prediction"]
    == holdout_predictions_frame["weak_sentiment_label"]
)

per_app_metric_rows = []
per_app_class_rows = []
for app_name in HELD_OUT_APPS:
    app_mask = holdout_predictions_frame["app_name"].eq(app_name)
    app_true = holdout_predictions_frame.loc[
        app_mask, "weak_sentiment_label"
    ]
    app_pred = holdout_predictions_frame.loc[
        app_mask, "strict_holdout_prediction"
    ]
    per_app_metric_rows.append(
        metric_row("strict_two_app_holdout", app_name, app_true, app_pred)
    )
    per_app_class_rows.extend(
        class_metric_rows(
            "strict_two_app_holdout",
            app_name,
            app_true,
            app_pred,
        )
    )

holdout_per_app_metrics = pd.DataFrame(per_app_metric_rows)
holdout_per_app_class_metrics = pd.DataFrame(per_app_class_rows)

holdout_confusion = pd.DataFrame(
    confusion_matrix(
        holdout_test["weak_sentiment_label"],
        strict_holdout_predictions,
        labels=LABEL_ORDER,
    ),
    index=[f"actual_{label}" for label in LABEL_ORDER],
    columns=[f"predicted_{label}" for label in LABEL_ORDER],
)

holdout_design_summary = pd.DataFrame(
    [
        {
            "design": "strict_app_and_text_group_holdout",
            "training_apps": strict_holdout_train["app_name"].nunique(),
            "training_rows": len(strict_holdout_train),
            "test_apps": holdout_test["app_name"].nunique(),
            "test_rows": len(holdout_test),
            "shared_text_groups": strict_overlap_groups,
            "training_rows_removed_for_shared_text": (
                len(app_only_train) - len(strict_holdout_train)
            ),
            "test_rows_in_shared_text_groups": 0,
        },
        {
            "design": "app_only_holdout_with_disclosed_text_overlap",
            "training_apps": app_only_train["app_name"].nunique(),
            "training_rows": len(app_only_train),
            "test_apps": holdout_test["app_name"].nunique(),
            "test_rows": len(holdout_test),
            "shared_text_groups": app_only_overlap_groups,
            "training_rows_removed_for_shared_text": 0,
            "test_rows_in_shared_text_groups": app_only_overlap_test_rows,
        },
    ]
)

holdout_metrics.to_csv(
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_metrics_v1.csv",
    index=False,
)
holdout_class_metrics.to_csv(
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_class_metrics_v1.csv",
    index=False,
)
holdout_per_app_metrics.to_csv(
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_per_app_metrics_v1.csv",
    index=False,
)
holdout_per_app_class_metrics.to_csv(
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_per_app_class_metrics_v1.csv",
    index=False,
)
holdout_confusion.to_csv(
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_confusion_matrix_v1.csv"
)
holdout_design_summary.to_csv(
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_design_v1.csv",
    index=False,
)

print(holdout_design_summary.to_string(index=False))
print()
print(holdout_metrics.round(6).to_string(index=False))
print()
print(holdout_per_app_metrics.round(6).to_string(index=False))
print()
print(holdout_per_app_class_metrics.round(6).to_string(index=False))


                                      design  training_apps  training_rows  test_apps  test_rows  shared_text_groups  training_rows_removed_for_shared_text  test_rows_in_shared_text_groups
           strict_app_and_text_group_holdout              8          20749          2       7794                   0                                   6058                                0
app_only_holdout_with_disclosed_text_overlap              8          26807          2       7794                 415                                      0                             1913

                 experiment                                        model  accuracy  balanced_accuracy  macro_f1  weighted_f1  test_rows
            two_app_holdout            strict_app_and_text_group_holdout  0.775340           0.556953  0.561145     0.772470       7794
two_app_holdout_sensitivity app_only_holdout_with_disclosed_text_overlap  0.771106           0.556485  0.559816     0.769748       7794

            experiment 

## 5. Validation and interpretation

Validation below checks source continuity, exact baseline reproduction, manual
sample design, annotation completeness, and strict holdout isolation. The
conclusion then prioritizes the next work by the evidence from all three parts.


In [6]:
prior_expected_metrics = {
    "accuracy": 0.8208092485549133,
    "balanced_accuracy": 0.5867184442494217,
    "macro_f1": 0.5898858850971797,
    "weighted_f1": 0.8196257837914523,
}
reproduced_full = feature_comparison_metrics.loc[
    feature_comparison_metrics["model"].eq(
        "tfidf_plus_engineered_features"
    )
].iloc[0]

validation_records = []


def add_validation(check, passed, observed):
    validation_records.append(
        {
            "check": check,
            "status": "passed" if bool(passed) else "failed",
            "passed": bool(passed),
            "observed": observed,
        }
    )


add_validation("merged_rows_equal_34601", len(data) == 34601, len(data))
add_validation(
    "baseline_train_rows_equal_27681",
    len(train_df) == 27681,
    len(train_df),
)
add_validation(
    "baseline_test_rows_equal_6920",
    len(test_df) == 6920,
    len(test_df),
)
add_validation(
    "baseline_text_group_overlap_zero",
    baseline_text_overlap == 0,
    baseline_text_overlap,
)
for metric_name, expected_value in prior_expected_metrics.items():
    add_validation(
        f"reproduced_baseline_{metric_name}",
        np.isclose(
            reproduced_full[metric_name],
            expected_value,
            atol=1e-12,
        ),
        reproduced_full[metric_name],
    )
add_validation(
    "baseline_error_count_equal_1240",
    len(error_pool) == 1240,
    len(error_pool),
)
add_validation(
    "manual_sample_rows_equal_120",
    len(manual_sample) == 120,
    len(manual_sample),
)
add_validation(
    "manual_sample_review_keys_unique",
    manual_sample["review_key"].is_unique,
    int(manual_sample["review_key"].duplicated().sum()),
)
core_counts = (
    manual_sample.loc[manual_sample["sample_layer"].eq("core_balanced")]
    ["weak_sentiment_label"]
    .value_counts()
    .reindex(LABEL_ORDER, fill_value=0)
)
add_validation(
    "manual_core_has_30_per_weak_label",
    core_counts.eq(30).all(),
    core_counts.to_dict(),
)
extra_neutral_count = int(
    (
        manual_sample["sample_layer"].eq("neutral_oversample")
        & manual_sample["weak_sentiment_label"].eq("neutral")
    ).sum()
)
add_validation(
    "manual_extra_neutral_rows_equal_30",
    extra_neutral_count == 30,
    extra_neutral_count,
)
add_validation(
    "manual_annotations_complete",
    not annotated_errors["john_primary_category"].isna().any(),
    int(annotated_errors["john_primary_category"].isna().sum()),
)
add_validation(
    "manual_annotation_review_keys_match_sample",
    len(annotated_errors) == 120,
    len(annotated_errors),
)
add_validation(
    "heldout_apps_absent_from_strict_training",
    not strict_holdout_train["app_name"].isin(HELD_OUT_APPS).any(),
    sorted(
        set(strict_holdout_train["app_name"]).intersection(HELD_OUT_APPS)
    ),
)
add_validation(
    "strict_holdout_text_group_overlap_zero",
    strict_overlap_groups == 0,
    strict_overlap_groups,
)
add_validation(
    "strict_holdout_training_apps_equal_8",
    strict_holdout_train["app_name"].nunique() == 8,
    strict_holdout_train["app_name"].nunique(),
)
add_validation(
    "strict_holdout_test_rows_equal_7794",
    len(holdout_test) == 7794,
    len(holdout_test),
)
add_validation(
    "all_labels_present_in_strict_holdout_train",
    set(strict_holdout_train["weak_sentiment_label"]) == set(LABEL_ORDER),
    sorted(strict_holdout_train["weak_sentiment_label"].unique()),
)
add_validation(
    "all_labels_present_in_strict_holdout_test",
    set(holdout_test["weak_sentiment_label"]) == set(LABEL_ORDER),
    sorted(holdout_test["weak_sentiment_label"].unique()),
)
add_validation(
    "strict_holdout_confusion_total_matches_test",
    int(holdout_confusion.to_numpy().sum()) == len(holdout_test),
    int(holdout_confusion.to_numpy().sum()),
)
metric_frames = [
    feature_comparison_metrics[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ].to_numpy(),
    holdout_metrics[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ].to_numpy(),
    holdout_per_app_metrics[
        ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    ].to_numpy(),
]
all_metrics_finite = all(np.isfinite(values).all() for values in metric_frames)
add_validation(
    "all_reported_metrics_finite",
    all_metrics_finite,
    "finite" if all_metrics_finite else "non-finite value found",
)

validation_checks = pd.DataFrame(validation_records)
if not validation_checks["passed"].all():
    failed = validation_checks.loc[~validation_checks["passed"]]
    raise ValueError("Validation failed:\n" + failed.to_string(index=False))

validation_checks.to_csv(
    OUTPUT_DIR / "weak_sentiment_error_analysis_validation_checks_v1.csv",
    index=False,
)

tfidf_metrics = feature_comparison_metrics.loc[
    feature_comparison_metrics["model"].eq("tfidf_only")
].iloc[0]
full_metrics = reproduced_full
tfidf_neutral_f1 = feature_comparison_class_metrics.loc[
    feature_comparison_class_metrics["model"].eq("tfidf_only")
    & feature_comparison_class_metrics["class"].eq("neutral"),
    "f1_score",
].iloc[0]
full_neutral_f1 = feature_comparison_class_metrics.loc[
    feature_comparison_class_metrics["model"].eq(
        "tfidf_plus_engineered_features"
    )
    & feature_comparison_class_metrics["class"].eq("neutral"),
    "f1_score",
].iloc[0]
strict_holdout_combined = holdout_metrics.loc[
    holdout_metrics["model"].eq("strict_app_and_text_group_holdout")
].iloc[0]
strict_holdout_neutral_f1 = holdout_class_metrics.loc[
    holdout_class_metrics["model"].eq(
        "strict_app_and_text_group_holdout"
    )
    & holdout_class_metrics["class"].eq("neutral"),
    "f1_score",
].iloc[0]

neutral_manual = annotated_errors.loc[
    annotated_errors["weak_sentiment_label"].eq("neutral")
]
neutral_primary_counts = (
    neutral_manual["john_primary_category"]
    .value_counts()
    .reindex(JOHN_CATEGORY_ORDER, fill_value=0)
)

metadata = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_stage": "weak_sentiment_error_analysis_and_transfer_v1",
    "scope": "exploratory_not_production",
    "source_rows": int(len(data)),
    "apps": int(data["app_name"].nunique()),
    "random_state": RANDOM_STATE,
    "baseline_split": {
        "train_rows": int(len(train_df)),
        "test_rows": int(len(test_df)),
        "text_group_overlap": int(baseline_text_overlap),
    },
    "manual_review": {
        "baseline_errors": int(len(error_pool)),
        "reviewed_rows": int(len(annotated_errors)),
        "balanced_core_rows": 90,
        "balanced_core_rows_per_weak_label": 30,
        "additional_neutral_rows": 30,
        "reviewers": 1,
        "adjudication": False,
        "primary_categories": JOHN_CATEGORY_ORDER,
        "sampling_warning": (
            "Counts describe a direction-balanced and app-balanced error sample "
            "and are not population estimates."
        ),
    },
    "feature_comparison": {
        "tfidf_only_macro_f1": float(tfidf_metrics["macro_f1"]),
        "full_feature_macro_f1": float(full_metrics["macro_f1"]),
        "macro_f1_delta_full_minus_tfidf": float(
            full_metrics["macro_f1"] - tfidf_metrics["macro_f1"]
        ),
        "tfidf_only_neutral_f1": float(tfidf_neutral_f1),
        "full_feature_neutral_f1": float(full_neutral_f1),
        "neutral_f1_delta_full_minus_tfidf": float(
            full_neutral_f1 - tfidf_neutral_f1
        ),
    },
    "strict_two_app_holdout": {
        "held_out_apps": HELD_OUT_APPS,
        "training_apps": int(strict_holdout_train["app_name"].nunique()),
        "training_rows": int(len(strict_holdout_train)),
        "test_rows": int(len(holdout_test)),
        "training_rows_removed_for_shared_text": int(
            len(app_only_train) - len(strict_holdout_train)
        ),
        "train_test_text_group_overlap": int(strict_overlap_groups),
        "accuracy": float(strict_holdout_combined["accuracy"]),
        "balanced_accuracy": float(
            strict_holdout_combined["balanced_accuracy"]
        ),
        "macro_f1": float(strict_holdout_combined["macro_f1"]),
        "weighted_f1": float(strict_holdout_combined["weighted_f1"]),
        "neutral_f1": float(strict_holdout_neutral_f1),
    },
    "recommended_priority": [
        "label_design",
        "evaluation_setup",
        "feature_design",
    ],
    "validation_checks_passed": int(validation_checks["passed"].sum()),
    "validation_checks_failed": int((~validation_checks["passed"]).sum()),
}
metadata_path = (
    OUTPUT_DIR / "weak_sentiment_error_analysis_metadata_v1.json"
)
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

report_text = f"""# Google Play Weak Sentiment Error Analysis and Cross-App Transfer v1

## Objective

This phase investigates the rating-derived weak-sentiment baseline without changing the model family or tuning for a higher headline score. It addresses:

1. the composition of baseline errors, especially three-star reviews;
2. TF-IDF alone versus the current engineered feature set; and
3. transfer to two apps held out entirely from training.

## Continuity and controls

- Source reviews: {len(data):,}
- Apps: {data['app_name'].nunique()}
- Original grouped training rows: {len(train_df):,}
- Original grouped test rows: {len(test_df):,}
- Exact normalized-text groups shared in the original split: {baseline_text_overlap}
- Rating mapping retained: 1–2 negative, 3 neutral, 4–5 positive
- Direct star-rating expressions remain redacted in model text
- Model retained: class-weighted `LinearSVC`, fixed `C=1.0`
- TF-IDF retained: word unigrams and bigrams, maximum 6,000 features
- Hyperparameter search: none

## Manual error review

The full-feature baseline made {len(error_pool):,} errors on the 6,920-row test split. The manual review contains:

- 90-row balanced core: 30 errors from each weak-label class
- 30 additional three-star errors
- 120 unique reviewed errors in total
- equal sampling from each wrong prediction direction within each weak-label class
- app round-robin selection within each direction

One reviewer used the original cleaned review text and source rating. The output records a text category, rating-consistency assessment, John's requested primary category, and a short rationale for every sampled review.

| Primary category | Count |
|---|---:|
| Neutral | {int(manual_category_summary.set_index('john_primary_category').loc['neutral', 'review_count'])} |
| Mixed | {int(manual_category_summary.set_index('john_primary_category').loc['mixed', 'review_count'])} |
| Positive | {int(manual_category_summary.set_index('john_primary_category').loc['positive', 'review_count'])} |
| Negative | {int(manual_category_summary.set_index('john_primary_category').loc['negative', 'review_count'])} |
| Unclear | {int(manual_category_summary.set_index('john_primary_category').loc['unclear', 'review_count'])} |
| Inconsistent with rating | {int(manual_category_summary.set_index('john_primary_category').loc['inconsistent_with_rating', 'review_count'])} |

### Three-star focus

Among the 60 direction-balanced three-star errors:

- inconsistent with rating: {int(neutral_primary_counts['inconsistent_with_rating'])}
- mixed: {int(neutral_primary_counts['mixed'])}
- unclear: {int(neutral_primary_counts['unclear'])}
- cleanly neutral: {int(neutral_primary_counts['neutral'])}

The underlying text review categorized 21 of these 60 reviews as negative and 19 as positive; 13 were mixed and 7 were unclear. None was judged cleanly neutral. Because the sample intentionally balances error directions and apps, these counts diagnose error types but do not estimate their prevalence across all three-star reviews.

## TF-IDF-only versus current feature set

| Model | Accuracy | Balanced accuracy | Macro F1 | Weighted F1 | Neutral F1 |
|---|---:|---:|---:|---:|---:|
| TF-IDF only | {tfidf_metrics['accuracy']:.4f} | {tfidf_metrics['balanced_accuracy']:.4f} | {tfidf_metrics['macro_f1']:.4f} | {tfidf_metrics['weighted_f1']:.4f} | {tfidf_neutral_f1:.4f} |
| TF-IDF + engineered features | {full_metrics['accuracy']:.4f} | {full_metrics['balanced_accuracy']:.4f} | {full_metrics['macro_f1']:.4f} | {full_metrics['weighted_f1']:.4f} | {full_neutral_f1:.4f} |

Full-feature minus TF-IDF-only changes:

- accuracy: {(full_metrics['accuracy'] - tfidf_metrics['accuracy']):+.4f}
- balanced accuracy: {(full_metrics['balanced_accuracy'] - tfidf_metrics['balanced_accuracy']):+.4f}
- macro F1: {(full_metrics['macro_f1'] - tfidf_metrics['macro_f1']):+.4f}
- weighted F1: {(full_metrics['weighted_f1'] - tfidf_metrics['weighted_f1']):+.4f}
- neutral F1: {(full_neutral_f1 - tfidf_neutral_f1):+.4f}

The two models agree on {agreement_count:,} of 6,920 predictions ({agreement_count / len(comparison_predictions):.2%}). Engineered features fix 34 TF-IDF-only errors but change 36 correct TF-IDF-only predictions to errors, for a net loss of two correct predictions. The engineered feature set therefore does not provide a material overall improvement in this controlled split. Its small neutral-class gain is not enough to resolve the label ambiguity.

## Two-app holdout transfer

YouTube and DoorDash were selected before fitting to provide different domains, volumes, and collection-activity profiles while retaining all three weak-label classes.

Primary strict design:

- held-out apps absent from training: 2
- remaining training apps: {strict_holdout_train['app_name'].nunique()}
- training rows before shared-text removal: {len(app_only_train):,}
- training rows after shared-text removal: {len(strict_holdout_train):,}
- training rows removed because their exact text group appeared in held-out apps: {len(app_only_train) - len(strict_holdout_train):,}
- held-out test rows: {len(holdout_test):,}
- train/test exact text-group overlap: {strict_overlap_groups}

| Evaluation | Accuracy | Balanced accuracy | Macro F1 | Weighted F1 | Neutral F1 |
|---|---:|---:|---:|---:|---:|
| Original grouped same-app split | {full_metrics['accuracy']:.4f} | {full_metrics['balanced_accuracy']:.4f} | {full_metrics['macro_f1']:.4f} | {full_metrics['weighted_f1']:.4f} | {full_neutral_f1:.4f} |
| Strict two-app holdout | {strict_holdout_combined['accuracy']:.4f} | {strict_holdout_combined['balanced_accuracy']:.4f} | {strict_holdout_combined['macro_f1']:.4f} | {strict_holdout_combined['weighted_f1']:.4f} | {strict_holdout_neutral_f1:.4f} |

The strict holdout lowers accuracy by {(strict_holdout_combined['accuracy'] - full_metrics['accuracy']):+.4f} and macro F1 by {(strict_holdout_combined['macro_f1'] - full_metrics['macro_f1']):+.4f}. YouTube macro F1 is {holdout_per_app_metrics.loc[holdout_per_app_metrics['model'].eq('YouTube'), 'macro_f1'].iloc[0]:.4f}; DoorDash macro F1 is {holdout_per_app_metrics.loc[holdout_per_app_metrics['model'].eq('DoorDash'), 'macro_f1'].iloc[0]:.4f}. This confirms app-level transfer variation.

The disclosed app-only sensitivity run has macro F1 {holdout_metrics.loc[holdout_metrics['model'].eq('app_only_holdout_with_disclosed_text_overlap'), 'macro_f1'].iloc[0]:.4f}, close to the strict result. Shared generic text does not explain away the transfer gap in this test.

## Error-source judgment

### 1. Label design is the first priority

The largest actionable issue is the meaning of the target. In the focused three-star error sample, the text frequently expresses clear positive or negative sentiment, mixed sentiment, or insufficient information instead of a clean neutral class. The next label iteration should retain the source rating but rename the derived class as a rating group rather than treating three stars as verified neutral sentiment. A manually labeled validation set should keep text sentiment and rating consistency as separate fields.

### 2. Evaluation setup is the second priority

The two-app holdout produces a meaningful decline and different results by app. Future evaluation should retain:

- exact-text-group isolation;
- a same-distribution grouped split for continuity;
- one or more app-heldout tests;
- per-class and per-app metrics; and
- explicit reporting of neutral support and error composition.

### 3. Feature design is the third priority

The current engineered features produce only a negligible macro-F1 change relative to TF-IDF alone. They should not be treated as a demonstrated improvement. TF-IDF-only is the cleaner reference until label and evaluation design are improved. Later feature work should be tied to reviewed error types such as contrast language, negation, short ambiguous text, and non-English coverage.

## Recommended next step

Do not move to a more complex model yet. First create a manually labeled validation set that includes both correct and incorrect predictions and keeps three separate concepts:

1. source star rating;
2. text sentiment; and
3. rating-text consistency.

Then rerun the simple TF-IDF reference under both grouped same-distribution and app-heldout evaluation. Only after that should a new feature or model be accepted, and only if it improves the relevant class and transfer metrics consistently.

## Limitations

1. The 120-row review is a designed error sample rather than a random estimate of all reviews.
2. One reviewer performed the annotations and no independent adjudication was completed.
3. Manual interpretation is difficult for short, ambiguous, sarcastic, or non-English reviews.
4. Only two apps were held out in this transfer test.
5. Strict text-group isolation removes many generic repeated-text rows from training.
6. Results apply to the current 10-app snapshot and fixed model configuration.
7. Ratings remain weak labels and do not establish true sentiment.
"""

report_path = (
    REPORT_DIR
    / "google_play_weak_sentiment_error_analysis_v1_report.md"
)
report_path.write_text(report_text, encoding="utf-8")

readme_update_text = f"""## Weak sentiment error analysis and cross-app transfer v1

### Scope

This phase diagnoses the rating-derived weak-sentiment baseline without tuning for a higher headline score. It includes:

- a 120-row manual review of baseline errors with additional three-star coverage;
- TF-IDF-only versus TF-IDF plus current engineered features on the same grouped split; and
- a strict two-app holdout using YouTube and DoorDash with zero exact text-group overlap.

### Main findings

1. In the 60 reviewed three-star errors, {int(neutral_primary_counts['inconsistent_with_rating'])} were categorized as inconsistent with the rating-derived neutral label, {int(neutral_primary_counts['mixed'])} as mixed, {int(neutral_primary_counts['unclear'])} as unclear, and {int(neutral_primary_counts['neutral'])} as cleanly neutral.
2. The current engineered features changed macro F1 from {tfidf_metrics['macro_f1']:.4f} to {full_metrics['macro_f1']:.4f} and neutral F1 from {tfidf_neutral_f1:.4f} to {full_neutral_f1:.4f}. This is not a material overall gain.
3. Strictly holding out YouTube and DoorDash reduced macro F1 from {full_metrics['macro_f1']:.4f} to {strict_holdout_combined['macro_f1']:.4f}. YouTube and DoorDash also produced different app-level results.
4. The recommended priority is label design first, evaluation setup second, and feature design third.
5. A more complex model is not recommended until a manually labeled validation set separates star rating, text sentiment, and rating-text consistency.

### Main files

```text
inputs/
└── manual_error_annotations_v1.csv

notebooks/
└── Google_Play_Weak_Sentiment_Error_Analysis_v1.ipynb

outputs/
├── weak_sentiment_error_analysis_source_validation_v1.csv
├── weak_sentiment_feature_set_comparison_metrics_v1.csv
├── weak_sentiment_feature_set_class_metrics_v1.csv
├── weak_sentiment_feature_set_prediction_changes_v1.csv
├── weak_sentiment_manual_error_review_v1.csv
├── weak_sentiment_manual_error_category_summary_v1.csv
├── weak_sentiment_manual_error_by_weak_label_v1.csv
├── weak_sentiment_manual_text_by_weak_label_v1.csv
├── weak_sentiment_manual_rating_consistency_v1.csv
├── weak_sentiment_manual_error_by_direction_v1.csv
├── weak_sentiment_two_app_holdout_design_v1.csv
├── weak_sentiment_two_app_holdout_metrics_v1.csv
├── weak_sentiment_two_app_holdout_class_metrics_v1.csv
├── weak_sentiment_two_app_holdout_per_app_metrics_v1.csv
├── weak_sentiment_two_app_holdout_per_app_class_metrics_v1.csv
├── weak_sentiment_two_app_holdout_confusion_matrix_v1.csv
├── weak_sentiment_error_analysis_validation_checks_v1.csv
├── weak_sentiment_error_analysis_metadata_v1.json
└── weak_sentiment_error_analysis_output_manifest_v1.csv

reports/
├── google_play_weak_sentiment_error_analysis_v1_report.md
└── README_weak_sentiment_error_analysis_v1_update.md
```

### Reproduction

1. Keep `outputs/modeling_ready_weak_sentiment_v0.csv` and `outputs/review_features_v0.csv` in the repository.
2. Keep `inputs/manual_error_annotations_v1.csv` in the repository.
3. Open `notebooks/Google_Play_Weak_Sentiment_Error_Analysis_v1.ipynb`.
4. Run all cells in order.
5. The notebook verifies the upstream file hashes, reproduces the prior full-feature baseline, recreates the deterministic error sample, joins the fixed manual annotations, runs both feature-set models, runs the two-app holdout, validates every result, and regenerates the CSV, JSON, and Markdown deliverables.

### Interpretation guardrail

Manual category counts describe the designed error sample and are not estimates for all reviews. This remains exploratory work and is not a production sentiment system.
"""
readme_update_path = (
    REPORT_DIR / "README_weak_sentiment_error_analysis_v1_update.md"
)
readme_update_path.write_text(readme_update_text, encoding="utf-8")

print(validation_checks.to_string(index=False))
print()
print("Recommended priority: label design -> evaluation setup -> feature design")
print("Report:", report_path)
print("README update:", readme_update_path)


                                      check status  passed                                        observed
                    merged_rows_equal_34601 passed    True                                           34601
            baseline_train_rows_equal_27681 passed    True                                           27681
              baseline_test_rows_equal_6920 passed    True                                            6920
           baseline_text_group_overlap_zero passed    True                                               0
               reproduced_baseline_accuracy passed    True                                        0.820809
      reproduced_baseline_balanced_accuracy passed    True                                        0.586718
               reproduced_baseline_macro_f1 passed    True                                        0.589886
            reproduced_baseline_weighted_f1 passed    True                                        0.819626
            baseline_error_count_equa

## Conclusion

The evidence does not support moving immediately to a more complex model.
Three-star reviews do not behave like a clean neutral sentiment class in the
reviewed errors, the engineered features add almost no controlled-split gain,
and performance declines when two apps are completely unseen.

The next improvement should therefore be:

1. **labels first** — separate star rating, text sentiment, and consistency;
2. **evaluation second** — retain grouped and app-heldout testing; and
3. **features third** — add only error-motivated features after labels and
   evaluation are stable.


In [7]:
manifest_targets = [
    OUTPUT_DIR / "weak_sentiment_error_analysis_source_validation_v1.csv",
    OUTPUT_DIR / "weak_sentiment_feature_set_comparison_metrics_v1.csv",
    OUTPUT_DIR / "weak_sentiment_feature_set_class_metrics_v1.csv",
    OUTPUT_DIR / "weak_sentiment_feature_set_prediction_changes_v1.csv",
    OUTPUT_DIR / "weak_sentiment_manual_error_review_v1.csv",
    OUTPUT_DIR / "weak_sentiment_manual_error_category_summary_v1.csv",
    OUTPUT_DIR / "weak_sentiment_manual_error_by_weak_label_v1.csv",
    OUTPUT_DIR / "weak_sentiment_manual_text_by_weak_label_v1.csv",
    OUTPUT_DIR / "weak_sentiment_manual_rating_consistency_v1.csv",
    OUTPUT_DIR / "weak_sentiment_manual_error_by_direction_v1.csv",
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_design_v1.csv",
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_metrics_v1.csv",
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_class_metrics_v1.csv",
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_per_app_metrics_v1.csv",
    OUTPUT_DIR
    / "weak_sentiment_two_app_holdout_per_app_class_metrics_v1.csv",
    OUTPUT_DIR / "weak_sentiment_two_app_holdout_confusion_matrix_v1.csv",
    OUTPUT_DIR / "weak_sentiment_error_analysis_validation_checks_v1.csv",
    OUTPUT_DIR / "weak_sentiment_error_analysis_metadata_v1.json",
    report_path,
    readme_update_path,
]

manifest_rows = []
for file_path in manifest_targets:
    if not file_path.exists() or file_path.stat().st_size == 0:
        raise FileNotFoundError(f"Required output missing or empty: {file_path}")
    manifest_rows.append(
        {
            "relative_path": str(file_path.relative_to(REPO_ROOT)),
            "size_bytes": file_path.stat().st_size,
            "sha256": sha256_file(file_path),
        }
    )

output_manifest = pd.DataFrame(manifest_rows)
output_manifest_path = (
    OUTPUT_DIR / "weak_sentiment_error_analysis_output_manifest_v1.csv"
)
output_manifest.to_csv(output_manifest_path, index=False)

print("All validation checks passed.")
print(f"Validation checks: {validation_checks['passed'].sum()}")
print(f"Manual reviews: {len(annotated_errors):,}")
print(
    "Macro F1 TF-IDF/full/strict holdout:",
    f"{tfidf_metrics['macro_f1']:.4f} /",
    f"{full_metrics['macro_f1']:.4f} /",
    f"{strict_holdout_combined['macro_f1']:.4f}",
)
print("Output manifest:", output_manifest_path)


All validation checks passed.
Validation checks: 23
Manual reviews: 120
Macro F1 TF-IDF/full/strict holdout: 0.5892 / 0.5899 / 0.5611
Output manifest: /workspace/scratch/b0526b08776c/phase3_work/build/google_play_weak_sentiment_error_analysis_v1_complete_github_package/outputs/weak_sentiment_error_analysis_output_manifest_v1.csv
